In [4]:
import pandas as pd, numpy as np, sqlite3
from pathlib import Path
# locate DB
candidates = [
    Path.cwd() / 'backtest_results.db',
    Path.cwd().parent / 'backtest_results.db',
    Path.cwd().parent.parent / 'src' / 'backtest_results.db',
    Path('/home/chris/backtester/dema_strat/src/backtest_results.db')
]
DB_PATH = next(p for p in candidates if p.exists())
con = sqlite3.connect(DB_PATH)
f0_be_exit = pd.read_sql('SELECT * FROM f0_smoothing_be', con)
smooth = pd.read_sql('SELECT * FROM f0_raw_smoothing', con)
chandelier = pd.read_sql('SELECT * FROM f0_smoothing_chandelier_exit', con)
con.close()
metric_cols = ['expectancy','win_rate','max_dd','sharpe_ratio','profit_factor','num_trades','return_pct']
def mean_summary(df):
    return df[metric_cols].mean().rename(lambda x: f'{x}_mean')
# compute per‑strategy means
summary_f0 = mean_summary(f0_be_exit)
summary_smooth = mean_summary(smooth)
summary_chand = mean_summary(chandelier)
combined = pd.concat([summary_f0, summary_smooth, summary_chand], axis=1)
combined.columns = ['be','smooth','chandelier']
# order by expectancy then win_rate (both descending)
ordering = combined.loc[['expectancy_mean','win_rate_mean']].mean().sort_values(ascending=False).index
print('combined means ordered by avg(expectancy, win_rate):')
print(combined[ordering])


combined means ordered by avg(expectancy, win_rate):
                          smooth            be    chandelier
expectancy_mean        -0.027509     -0.024754     -0.028243
win_rate_mean          45.253726     42.380837     39.414170
max_dd_mean             5.144305      1.975503      4.060477
sharpe_ratio_mean     -37.950990    -12.341416    -27.469253
profit_factor_mean      0.568887      0.694056      0.594871
num_trades_mean     89295.503556  36947.781333  70276.677037
return_pct_mean        -5.139960     -1.960574     -4.055824
